# Psychiatric Comorbidities in Epilepsy — Visualizations

**Contents:**
1. Bar chart: Prevalence comparison (Surgical vs Non-Surgical)
2. Grouped bar chart: Detailed psychiatric categories
3. Temporal trends: Psychiatric comorbidity prevalence over time
4. Publication-ready Table 1

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.stats import chi2_contingency
import duckdb
import sys
sys.path.insert(0, '.')
from icd_codes import PSYCH_CATEGORIES, EPILEPSY_SURGERY_ICD10_PCS, EPILEPSY_SURGERY_ICD9

plt.rcParams.update({
    'font.size': 11,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150,
})

PROJ_DIR = '/Volumes/Niels 2/MIMIC/physionet.org/files/mimiciv/3.1/analysis/epilepsy_psych'
DB_PATH = '/Volumes/Niels 2/MIMIC/databases/mimic4.db'

# Load previously saved data
cohort = pd.read_csv(f'{PROJ_DIR}/epilepsy_cohort.csv', parse_dates=['admittime', 'dischtime'])
prev_df = pd.read_csv(f'{PROJ_DIR}/prevalence_results.csv')

print(f'Cohort: {len(cohort):,} admissions, {cohort["subject_id"].nunique():,} patients')
print(f'Surgical: {cohort["had_epilepsy_surgery"].sum():,} admissions')

## 1. Overall Prevalence Bar Chart (Surgical vs Non-Surgical)

In [ ]:
# Filter to main categories (exclude PNES and organic psych, keep 'Any')
display_cats = [
    'Any Psychiatric Disorder',
    'Depressive Disorders',
    'Substance Use Disorders',
    'Anxiety Disorders',
    'Bipolar Disorders',
    'Schizophrenia / Psychotic Disorders',
    'PTSD / Trauma-Related',
    'Suicidal Ideation',
    'ADHD',
    'Obsessive-Compulsive Disorders',
]

plot_df = prev_df[prev_df['category'].isin(display_cats)].copy()

# Maintain order
plot_df['category'] = pd.Categorical(plot_df['category'], categories=display_cats, ordered=True)
plot_df = plot_df.sort_values('category')

fig, ax = plt.subplots(figsize=(10, 6))

cats = display_cats
x = np.arange(len(cats))
width = 0.35

nonsurg = plot_df[plot_df['group'] == 'Non-Surgical'].set_index('category').reindex(cats)
surg = plot_df[plot_df['group'] == 'Surgical'].set_index('category').reindex(cats)

bars1 = ax.barh(x + width/2, nonsurg['prevalence_pct'], width, 
                label=f'Non-Surgical (n={nonsurg["n_total"].iloc[0]:,})', 
                color='#4C72B0', alpha=0.85)
bars2 = ax.barh(x - width/2, surg['prevalence_pct'], width,
                label=f'Surgical (n={surg["n_total"].iloc[0]:,})',
                color='#DD8452', alpha=0.85)

ax.set_yticks(x)
ax.set_yticklabels(cats)
ax.set_xlabel('Prevalence (%)')
ax.set_title('Psychiatric Comorbidity Prevalence in Epilepsy Patients\nSurgical vs Non-Surgical (MIMIC-IV)', 
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.invert_yaxis()

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        w = bar.get_width()
        if w > 0:
            ax.text(w + 0.5, bar.get_y() + bar.get_height()/2, f'{w:.1f}%',
                    va='center', fontsize=8)

plt.tight_layout()
plt.savefig(f'{PROJ_DIR}/fig1_prevalence_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig1_prevalence_comparison.png')

## 2. Chi-Square Tests: Statistical Comparison

In [ ]:
# Chi-square test for each category
stat_results = []

psych_cols = [c for c in cohort.columns if c.startswith('has_')]
# Also test 'any_psych'
test_cols = psych_cols + ['any_psych']

for col in test_cols:
    if col not in cohort.columns:
        continue
    ct = pd.crosstab(cohort['had_epilepsy_surgery'], cohort[col])
    if ct.shape == (2, 2):
        chi2, p, dof, expected = chi2_contingency(ct)
    elif ct.shape[1] == 1:
        # Only one column (all 0 or all 1 in one group)
        chi2, p = 0, 1.0
    else:
        continue
    
    # Get category label
    cat_key = col.replace('has_', '')
    if cat_key == 'any_psych':
        label = 'Any Psychiatric Disorder'
    else:
        label = PSYCH_CATEGORIES.get(cat_key, {}).get('label', cat_key)
    
    # Prevalences
    nonsurg = cohort[cohort['had_epilepsy_surgery'] == 0]
    surg = cohort[cohort['had_epilepsy_surgery'] == 1]
    
    p_nonsurg = nonsurg[col].mean() * 100
    p_surg = surg[col].mean() * 100
    
    stat_results.append({
        'Disorder': label,
        'Non-Surgical %': round(p_nonsurg, 1),
        'Surgical %': round(p_surg, 1),
        'Chi2': round(chi2, 2),
        'p-value': f'{p:.4f}' if p >= 0.0001 else '<0.0001',
        'Significant': '*' if p < 0.05 else '',
    })

stats_df = pd.DataFrame(stat_results)
stats_df = stats_df.sort_values('Non-Surgical %', ascending=False)
print('=== Chi-Square Tests: Surgical vs Non-Surgical ===')
stats_df

## 3. Temporal Trends

In [ ]:
# Extract admission year
cohort['admit_year'] = cohort['admittime'].dt.year

# Check year range
print(f'Year range: {cohort["admit_year"].min()} - {cohort["admit_year"].max()}')
print(f'Admissions per year:')
print(cohort['admit_year'].value_counts().sort_index())

In [ ]:
# Temporal trend: prevalence of key psychiatric categories by year
key_categories = {
    'any_psych': 'Any Psychiatric Disorder',
    'has_depression': 'Depression',
    'has_anxiety': 'Anxiety',
    'has_substance_use': 'Substance Use',
    'has_bipolar': 'Bipolar',
    'has_psychotic': 'Psychosis',
}

# Group by year, compute prevalence
years = sorted(cohort['admit_year'].unique())

trend_data = []
for year in years:
    yr_sub = cohort[cohort['admit_year'] == year]
    n = len(yr_sub)
    if n < 10:  # skip years with very few admissions
        continue
    for col, label in key_categories.items():
        if col in yr_sub.columns:
            prev = yr_sub[col].mean() * 100
            trend_data.append({'year': year, 'category': label, 'prevalence': prev, 'n': n})

trend_df = pd.DataFrame(trend_data)

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#333333', '#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3']
for i, (col, label) in enumerate(key_categories.items()):
    cat_trend = trend_df[trend_df['category'] == label]
    style = '-' if label == 'Any Psychiatric Disorder' else '--'
    lw = 2.5 if label == 'Any Psychiatric Disorder' else 1.5
    ax.plot(cat_trend['year'], cat_trend['prevalence'], style, 
            label=label, color=colors[i], linewidth=lw, marker='o', markersize=4)

ax.set_xlabel('Admission Year')
ax.set_ylabel('Prevalence (%)')
ax.set_title('Temporal Trends in Psychiatric Comorbidity Among Epilepsy Patients\n(MIMIC-IV)',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PROJ_DIR}/fig2_temporal_trends.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig2_temporal_trends.png')

In [ ]:
# Temporal trend by surgical status (any psych)
fig, ax = plt.subplots(figsize=(10, 5))

for sflag, label, color in [(0, 'Non-Surgical', '#4C72B0'), (1, 'Surgical', '#DD8452')]:
    sub = cohort[cohort['had_epilepsy_surgery'] == sflag]
    yearly = sub.groupby('admit_year').agg(
        n=('any_psych', 'count'),
        prev=('any_psych', 'mean')
    ).reset_index()
    yearly = yearly[yearly['n'] >= 5]  # min 5 admissions
    yearly['prev'] = yearly['prev'] * 100
    
    ax.plot(yearly['admit_year'], yearly['prev'], '-o', label=label, 
            color=color, markersize=4, linewidth=2)

ax.set_xlabel('Admission Year')
ax.set_ylabel('Prevalence of Any Psychiatric Comorbidity (%)')
ax.set_title('Any Psychiatric Comorbidity Over Time\nSurgical vs Non-Surgical Epilepsy (MIMIC-IV)',
             fontsize=13, fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PROJ_DIR}/fig3_temporal_surg_vs_nonsurg.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig3_temporal_surg_vs_nonsurg.png')

## 4. Stacked Bar Chart: Psychiatric Comorbidity Burden

In [ ]:
# How many psychiatric categories does each patient have?
psych_count_cols = [c for c in cohort.columns if c.startswith('has_') and c != 'has_pnes']
cohort['n_psych_categories'] = cohort[psych_count_cols].sum(axis=1)

# Bin: 0, 1, 2, 3+
cohort['psych_burden'] = cohort['n_psych_categories'].apply(
    lambda x: '0' if x == 0 else ('1' if x == 1 else ('2' if x == 2 else '3+'))
)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

for i, (sflag, title) in enumerate([(0, 'Non-Surgical'), (1, 'Surgical')]):
    sub = cohort[cohort['had_epilepsy_surgery'] == sflag]
    counts = sub['psych_burden'].value_counts(normalize=True).reindex(['0', '1', '2', '3+']).fillna(0) * 100
    
    colors = ['#95a5a6', '#f39c12', '#e74c3c', '#8e44ad']
    bars = axes[i].bar(counts.index, counts.values, color=colors, edgecolor='white')
    axes[i].set_title(f'{title}\n(n={len(sub):,})', fontweight='bold')
    axes[i].set_xlabel('Number of Psychiatric Comorbidity Categories')
    axes[i].set_ylabel('Percentage of Admissions (%)')
    axes[i].set_ylim(0, 75)
    
    for bar, val in zip(bars, counts.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{val:.1f}%', ha='center', fontsize=9)

fig.suptitle('Psychiatric Comorbidity Burden in Epilepsy Patients', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PROJ_DIR}/fig4_psych_burden.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig4_psych_burden.png')

## 5. Publication-Ready Table 1: Demographics & Clinical Characteristics

In [ ]:
def make_table1(cohort):
    """Generate Table 1 comparing surgical vs non-surgical."""
    rows = []
    
    nonsurg = cohort[cohort['had_epilepsy_surgery'] == 0]
    surg = cohort[cohort['had_epilepsy_surgery'] == 1]
    
    # N
    rows.append(('N (admissions)', f'{len(nonsurg):,}', f'{len(surg):,}', ''))
    rows.append(('N (unique patients)', f'{nonsurg["subject_id"].nunique():,}', f'{surg["subject_id"].nunique():,}', ''))
    
    # Age
    rows.append(('Age, mean (SD)', 
                 f'{nonsurg["anchor_age"].mean():.1f} ({nonsurg["anchor_age"].std():.1f})',
                 f'{surg["anchor_age"].mean():.1f} ({surg["anchor_age"].std():.1f})', ''))
    
    # Gender
    for g in ['F', 'M']:
        n_ns = (nonsurg['gender'] == g).sum()
        n_s = (surg['gender'] == g).sum()
        rows.append((f'Gender: {g}',
                     f'{n_ns:,} ({n_ns/len(nonsurg)*100:.1f}%)',
                     f'{n_s:,} ({n_s/len(surg)*100:.1f}%)', ''))
    
    # Race (top categories)
    for race in ['WHITE', 'BLACK/AFRICAN AMERICAN', 'HISPANIC/LATINO', 'ASIAN', 'OTHER']:
        n_ns = (nonsurg['race'] == race).sum()
        n_s = (surg['race'] == race).sum()
        pct_ns = n_ns/len(nonsurg)*100 if len(nonsurg) > 0 else 0
        pct_s = n_s/len(surg)*100 if len(surg) > 0 else 0
        rows.append((f'Race: {race.title()}',
                     f'{n_ns:,} ({pct_ns:.1f}%)',
                     f'{n_s:,} ({pct_s:.1f}%)', ''))
    
    # Insurance
    for ins in cohort['insurance'].value_counts().head(4).index:
        n_ns = (nonsurg['insurance'] == ins).sum()
        n_s = (surg['insurance'] == ins).sum()
        rows.append((f'Insurance: {ins.title()}',
                     f'{n_ns:,} ({n_ns/len(nonsurg)*100:.1f}%)',
                     f'{n_s:,} ({n_s/len(surg)*100:.1f}%)', ''))
    
    # LOS
    rows.append(('LOS days, median (IQR)',
                 f'{nonsurg["los_days"].median():.0f} ({nonsurg["los_days"].quantile(0.25):.0f}-{nonsurg["los_days"].quantile(0.75):.0f})',
                 f'{surg["los_days"].median():.0f} ({surg["los_days"].quantile(0.25):.0f}-{surg["los_days"].quantile(0.75):.0f})', ''))
    
    # Mortality
    mort_ns = nonsurg['hospital_expire_flag'].mean() * 100
    mort_s = surg['hospital_expire_flag'].mean() * 100
    rows.append(('In-hospital mortality',
                 f'{nonsurg["hospital_expire_flag"].sum():,} ({mort_ns:.1f}%)',
                 f'{surg["hospital_expire_flag"].sum():,} ({mort_s:.1f}%)', ''))
    
    # Psychiatric comorbidities
    rows.append(('--- Psychiatric Comorbidities ---', '', '', ''))
    
    psych_map = {
        'any_psych': 'Any psychiatric disorder',
        'has_depression': 'Depressive disorders',
        'has_anxiety': 'Anxiety disorders',
        'has_substance_use': 'Substance use disorders',
        'has_bipolar': 'Bipolar disorders',
        'has_psychotic': 'Psychotic disorders',
        'has_ptsd': 'PTSD',
        'has_suicidal_ideation': 'Suicidal ideation',
        'has_adhd': 'ADHD',
        'has_ocd': 'OCD',
    }
    
    for col, label in psych_map.items():
        if col in cohort.columns:
            n_ns = nonsurg[col].sum()
            n_s = surg[col].sum()
            # Chi-square p-value
            ct = pd.crosstab(cohort['had_epilepsy_surgery'], cohort[col])
            if ct.shape == (2, 2):
                _, p, _, _ = chi2_contingency(ct)
                p_str = f'{p:.4f}' if p >= 0.0001 else '<0.0001'
            else:
                p_str = 'N/A'
            rows.append((label,
                         f'{n_ns:,} ({n_ns/len(nonsurg)*100:.1f}%)',
                         f'{n_s:,} ({n_s/len(surg)*100:.1f}%)',
                         p_str))
    
    table1 = pd.DataFrame(rows, columns=['Variable', 'Non-Surgical', 'Surgical', 'p-value'])
    return table1

table1 = make_table1(cohort)
table1

In [ ]:
# Save Table 1
table1.to_csv(f'{PROJ_DIR}/table1_demographics.csv', index=False)
print('Saved: table1_demographics.csv')

## 6. How to Report Psychiatric Diseases

### Recommended Reporting Structure for Publication

**Group psychiatric diagnoses into these clinically meaningful categories:**

| Category | ICD Codes Included | Report As |
|----------|-------------------|----------|
| **Mood Disorders** | Depression (F32-F33) + Bipolar (F31) | Combined and separately |
| **Anxiety Disorders** | GAD, panic, phobias (F40-F41) | Combined |
| **Trauma-Related** | PTSD (F43.1) | Separately |
| **Psychotic Disorders** | Schizophrenia spectrum (F20-F29) + Organic psychosis (F06.0, F06.2) | Combined — note organic psychosis is especially relevant in epilepsy |
| **Substance Use** | All SUDs (F10-F19) | Combined, with alcohol vs other breakdown |
| **ADHD** | F90 | Separately |
| **Suicidal Ideation** | R45.851 | Separately — important safety outcome |
| **Any Psychiatric Disorder** | Any of the above | Primary outcome |

### Reporting Tips:
1. **Primary outcome**: Report "any psychiatric comorbidity" prevalence first
2. **Individual categories**: Then break down by category, ordered by prevalence
3. **n/N (%)** format: Always report as "n/N (X.X%)" — e.g., "5,149/20,064 (25.7%)"
4. **Confidence intervals**: Add 95% CIs for prevalence estimates
5. **Comparison**: Chi-square or Fisher's exact test for surgical vs non-surgical
6. **Adjusted analysis**: Logistic regression adjusting for age, sex, race, insurance
7. **Note the limitation**: Administrative coding underestimates true psychiatric prevalence — your numbers are lower bounds

In [ ]:
# Forest plot style: Prevalence with 95% CI
from scipy.stats import norm

def wilson_ci(n_success, n_total, alpha=0.05):
    """Wilson score confidence interval for a proportion."""
    if n_total == 0:
        return 0, 0, 0
    p = n_success / n_total
    z = norm.ppf(1 - alpha/2)
    denom = 1 + z**2 / n_total
    center = (p + z**2 / (2*n_total)) / denom
    margin = z * np.sqrt((p*(1-p) + z**2/(4*n_total)) / n_total) / denom
    return p*100, max(0, (center - margin)*100), min(100, (center + margin)*100)

# Build forest plot data
forest_cats = [
    ('any_psych', 'Any Psychiatric Disorder'),
    ('has_depression', 'Depressive Disorders'),
    ('has_substance_use', 'Substance Use Disorders'),
    ('has_anxiety', 'Anxiety Disorders'),
    ('has_bipolar', 'Bipolar Disorders'),
    ('has_psychotic', 'Psychotic Disorders'),
    ('has_ptsd', 'PTSD'),
    ('has_suicidal_ideation', 'Suicidal Ideation'),
    ('has_adhd', 'ADHD'),
    ('has_ocd', 'OCD'),
]

fig, ax = plt.subplots(figsize=(10, 7))

y_positions = np.arange(len(forest_cats))
offset = 0.15

for sflag, label, color, yoff in [(0, 'Non-Surgical', '#4C72B0', offset), 
                                   (1, 'Surgical', '#DD8452', -offset)]:
    sub = cohort[cohort['had_epilepsy_surgery'] == sflag]
    n = len(sub)
    
    for i, (col, cat_label) in enumerate(forest_cats):
        if col not in sub.columns:
            continue
        nw = sub[col].sum()
        prev, ci_lo, ci_hi = wilson_ci(nw, n)
        
        ax.errorbar(prev, i + yoff, xerr=[[prev-ci_lo], [ci_hi-prev]], 
                    fmt='o', color=color, markersize=6, capsize=3, 
                    label=label if i == 0 else None)

ax.set_yticks(y_positions)
ax.set_yticklabels([c[1] for c in forest_cats])
ax.set_xlabel('Prevalence (%) with 95% CI')
ax.set_title('Psychiatric Comorbidity Prevalence with 95% Confidence Intervals\nSurgical vs Non-Surgical Epilepsy (MIMIC-IV)',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PROJ_DIR}/fig5_forest_plot.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig5_forest_plot.png')

In [ ]:
print('\n=== Summary of Key Findings ===')
print(f'Total epilepsy cohort: {len(cohort):,} admissions ({cohort["subject_id"].nunique():,} patients)')
print(f'Surgical: {cohort["had_epilepsy_surgery"].sum():,} admissions')
print(f'Non-surgical: {(cohort["had_epilepsy_surgery"]==0).sum():,} admissions')
print()

nonsurg = cohort[cohort['had_epilepsy_surgery'] == 0]
surg = cohort[cohort['had_epilepsy_surgery'] == 1]
print(f'Any psych - Non-surgical: {nonsurg["any_psych"].mean()*100:.1f}%')
print(f'Any psych - Surgical: {surg["any_psych"].mean()*100:.1f}%')
print(f'Depression - Non-surgical: {nonsurg["has_depression"].mean()*100:.1f}%')
print(f'Depression - Surgical: {surg["has_depression"].mean()*100:.1f}%')